# Transformer Model Comparison

This notebook compares multiple pre-trained transformer models for sequence classification.
It trains and evaluates various models from the Hugging Face model hub on a labeled dataset.

In [ ]:
import pandas as pd
import evaluate
import numpy as np
import datetime
import os
import logging
import random
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, set_seed



### Seed Setting

In [ ]:
SEED = 42

def set_all_seeds(seed: int = SEED, deterministic: bool = False):
    """
    Set all random seeds for reproducibility across Python, NumPy, PyTorch, and Transformers.

    Args:
        seed: Random seed value to use for all random number generators.
        deterministic: If True, enables deterministic CUDA operations. This may reduce performance
                      but ensures full reproducibility on GPU. Note that some operations may not
                      have deterministic implementations.

    Returns:
        None
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    set_seed(seed)

    if deterministic:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(SEED, deterministic=False)

### Error Logging Setup

In [ ]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

log_file = os.path.join(log_dir, "error_log.csv")

logging.basicConfig(level=logging.ERROR,
                    format="%(asctime)s,%(levelname)s,%(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")

def log_error(error_message, model_name="Unknown"):
    """
    Log error messages to a CSV file with timestamp and model information.

    Args:
        error_message: The error message or exception details to log.
        model_name: Name or identifier of the model where the error occurred.

    Returns:
        None
    """

    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if not os.path.exists(log_file):
        df = pd.DataFrame(columns=["timestamp", "model_name", "error_message"])
        df.to_csv(log_file, index=False)

    df = pd.DataFrame([[timestamp, model_name, error_message]],
                      columns=["timestamp", "model_name", "error_message"])
    df.to_csv(log_file, mode="a", header=False, index=False)

### Paths and Relatives

In [ ]:
OUTPUT_DIR = "../models"
MODEL_NAME_LIST = [
        # 'AndreaSimeri/LegalBERT_GDPR',
        # 'studio-ousia/luke-base',
        # 'ahmedrachid/FinancialBERT',
        # 'google-bert/bert-base-uncased',
         'FacebookAI/roberta-base',
        # 'FacebookAI/xlm-roberta-base',
        # 'google/electra-base-discriminator',
        # 'distilbert/distilbert-base-uncased',
        # 'albert/albert-base-v2',
        # 'MoritzLaurer/DeBERTa-v3-xsmall-mnli-fever-anli-ling-binary',
        # 'google/electra-base-discriminator',
        # 'nlpaueb/legal-bert-base-uncased',
        # 'nlpaueb/bert-base-uncased-contracts',
        # 'nlpaueb/bert-base-uncased-eurlex',
        # 'nlpaueb/bert-base-uncased-echr',
]

DATA_PATH = "../data/input/ProcessDataset.csv"
DATA_ENCODING = "latin1"
DATA_SEP = ";"

LABEL_COL = "GDPR-criticality"
TEXT_COL = "Concatenation"


### Read Process Data Set

In [ ]:
raw_data = pd.read_csv(DATA_PATH, sep=DATA_SEP, encoding=DATA_ENCODING)
raw_data = raw_data.dropna(axis=1)
input_data = raw_data.copy(deep=True)

### Model Training and Evaluation

In [ ]:
for i in MODEL_NAME_LIST:
    MODEL_NAME = i
    SAVE_DIR = OUTPUT_DIR + '/' + MODEL_NAME.replace('/', '_') + '/'
    print('processed Model: ', i)
    try:
        input_data.rename(columns={TEXT_COL:'text', LABEL_COL:'labels'}, inplace=True)
        X_train, X_vtest, y_train, y_vtest = train_test_split(input_data['text'], input_data['labels'],
                                                            test_size=0.4,
                                                            random_state=SEED)
        X_test, X_val, y_test, y_val = train_test_split(X_vtest, y_vtest,
                                                       test_size=0.5,
                                                       random_state=SEED)
        datasets = DatasetDict({
            "train": Dataset.from_pandas(pd.concat([X_train, y_train], axis=1)),
            "test": Dataset.from_pandas(pd.concat([X_test, y_test], axis=1)),
            "val": Dataset.from_pandas(pd.concat([X_val, y_val], axis=1))
            })

        def preprocess_function(examples):
            """
            Tokenize text examples for transformer model input.

            Args:
                examples: Dictionary containing 'text' field with input texts.

            Returns:
                Dictionary with tokenized inputs (input_ids, attention_mask, etc.).
            """
            return tokenizer(examples['text'],
                             padding='max_length',
                             truncation=True,
                             max_length=512
                             )

        tokenizer = AutoTokenizer.from_pretrained(i)
        tokenized_data = datasets.map(preprocess_function, batched=True)

        if i in ['FacebookAI/roberta-base',
                 'FacebookAI/xlm-roberta-base',
                 'google/electra-base-discriminator']:

            model = AutoModelForSequenceClassification.from_pretrained(
                        MODEL_NAME,
                        num_labels=2,
                        torch_dtype="auto",
                        device_map = 'auto',
                        ignore_mismatched_sizes=True
            )

        else:
            model = AutoModelForSequenceClassification.from_pretrained(
                    MODEL_NAME,
                    num_labels=2,
                    ignore_mismatched_sizes=True
                )

            model.classifier = torch.nn.Linear(model.config.hidden_size, 2)

        metric = evaluate.load("recall")

        def compute_metrics(eval_pred):
            """
            Compute evaluation metrics from model predictions.

            Args:
                eval_pred: Tuple of (logits, labels) from model evaluation.

            Returns:
                Dictionary containing accuracy and recall scores.
            """
            logits, labels = eval_pred
            predictions = np.argmax(logits,
                                    axis=-1)
            return metric.compute(predictions=predictions,
                                  references=labels)

        training_args = TrainingArguments(output_dir=SAVE_DIR,
                                          eval_strategy="epoch",
                                          save_strategy="epoch",
                                          learning_rate=1e-5,
                                          num_train_epochs=3,
                                          load_best_model_at_end=True,
                                          metric_for_best_model="recall")#"accuracy") # former accuracy


        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_data['train'],
            eval_dataset=tokenized_data['val'],
            # data_collator=data_collator,
            compute_metrics=compute_metrics
        )

        trainer.train()

        trainer.save_model(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print('trainer saved')

        test_predictions = trainer.predict(tokenized_data['test'])
        test_predictions_argmax = np.argmax(test_predictions[0], axis=1)
        test_references = np.array(datasets["test"]["labels"])


        logits = test_predictions.predictions
        probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)  # Softmax anwenden
        y_probs = probs[:, 1]  # Wahrscheinlichkeiten für Klasse 1
        y_pred = np.argmax(logits, axis=-1)  # Klassenvorhersagen
        y_true = np.array(tokenized_data["test"]['labels'])

        report = classification_report(test_references, test_predictions_argmax, output_dict=True)

        precision = report['macro avg']['precision']
        recall = report['macro avg']['recall']
        f1 = report['macro avg']['f1-score']

        aucs = roc_auc_score(y_true, y_probs)
        bac = balanced_accuracy_score(y_true, test_predictions_argmax)
        acc = accuracy_score(y_true, test_predictions_argmax)

        precision_class_1 = report["1"]["precision"]
        recall_class_1 = report["1"]["recall"]
        f1_class_1 = report["1"]["f1-score"]
        precision_class_0 = report["0"]["precision"]
        recall_class_0 = report["0"]["recall"]
        f1_class_0 = report["0"]["f1-score"]


        probs = test_predictions.predictions
        probs = np.exp(probs) / np.exp(probs).sum(axis=1, keepdims=True)  # Softmax anwenden
        probs = probs[:, 1]  # Wahrscheinlichkeiten für Klasse 1

        # Wahre Labels
        y_true = np.array(tokenized_data["test"]['labels'])

        session_name = os.getenv("JPY_SESSION_NAME")

        df_results = pd.DataFrame({
            "TimeStamp": datetime.datetime.now(),
            "UsedNotebook": os.path.basename(session_name),
            "ModelName": MODEL_NAME,
            "Metric": ["Macro_BalancedAccuracy", "Macro_Precision", "Macro_Recall", "Macro_F1-Score", "AreaUnderTheCurve", "Micro_Precision_Crit", "Micro_Recall_Crit","Micro_F1_Crit", "Micro_Precision_UnCrit","Micro_Recall_UnCrit","Micro_F1_UnCrit"],
            "Value": [bac, precision, recall, f1, aucs, precision_class_1, recall_class_1, f1_class_1, precision_class_0, recall_class_0, f1_class_0]
        })

        # Ergebnisse ausgeben
        MODEL_NAME_SAVE = MODEL_NAME.replace('/', '_')
        df_results.to_csv('../data/output/transformer_comparison/results_'+MODEL_NAME_SAVE+'.csv', encoding='utf-8', index=False)

    except Exception as e:

        logging.error(f"Fehler: {str(e)}")
        log_error(str(e), model_name=MODEL_NAME)

        print("Something went wrong")
        pass
